## Transformation Pipeline

### Objectives

- Load Product Details, Retail Data 1, and Retail Data 2.
- Create a unified **Customers** table by combining customer data from both retail datasets.
- Apply PII masking to protect customer privacy.
- Create a consolidated **Sales** table containing:
  * Transaction details
  * Product ID
  * Customer ID
  * Other transaction-related attributes
- Remove all customer and product descriptive details from the Sales table.
- Add a column to classify quantities as **Valid** or **Invalid**.
- Standardize all dates to the format **yyyy-MM-dd**.
- Generate the final three tables:
  * **Products**
  * **Customers**
  * **Sales**


In [0]:
# Setting up the Path

# Set the storage account key in spark config
spark.conf.set(
    "fs.azure.account.key.retailsalesresources.blob.core.windows.net",
    dbutils.secrets.get("databricksScope", "storageaccount-secrets")  
)

# File path for raw-data container
RAW_DATA_PATH = "wasbs://raw-data@retailsalesresources.blob.core.windows.net"
TRANSFORMED_DATA_PATH = "wasbs://transformed-data@retailsalesresources.blob.core.windows.net"


try:
    # Loading the CSV files
    retail_data1 = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_DATA_PATH}/retail_data.csv")
    )

    retail_data2 = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_DATA_PATH}/retail_data2.csv")
    )

    product_details = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_DATA_PATH}/product_details.csv")
    )

    print("Datasets have been loaded successfuly!!")
except Exception as e:
    raise Exception("Couldn't load the CSV file")


Datasets have been loaded successfuly!!


In [0]:
EXPECTED_PRODUCT_DETAILS_COLS = 4
EXPECTED_RETAIL_DATA_COLS = 16

if product_details.count() == 0 or len(product_details.columns) != EXPECTED_PRODUCT_DETAILS_COLS:
    raise Exception("Failed to load Product details Table!")
print("Product details loaded successfully")

if retail_data1.count() == 0 or len(retail_data1.columns) != EXPECTED_RETAIL_DATA_COLS:
    raise Exception("Failed to load Retail data Table!")
print("Retail Data1 loaded successfully")

if retail_data2.count() == 0 or len(retail_data2.columns) != EXPECTED_RETAIL_DATA_COLS:
    raise Exception("Failed to load Retail data Table!")
print("Retail Data2 loaded successfully")

Product details loaded successfully
Retail Data1 loaded successfully
Retail Data2 loaded successfully


In [0]:
print("RETAIL DATA1\n")

print("Rows loaded: ", retail_data1.count())
retail_data1.limit(5).display()

RETAIL DATA1

Rows loaded:  4243


transaction_id,customer_id,customer_name,product_id,price,product_name,category,purchase_location,city,transaction_date,quantity,payment_method,discount,email,phone,payment_status
1,642,Troy Mitchell,108,15000,Mixer grinder,Home Appliances,offline,Bangalore,2025-12-06,3,UPI,0.1,Troy60@gmail.com,8385276968,successful
2,881,Sarah Guerrero,102,70000,Phone,ELEC,online,Bangalore,2025-07-21,5,Card,0.35,SarahGuerrero@yahoo.com,7147248911,successful
3,505,Samantha Hull,109,65000,Refrigerator,Home Appliances,offline,Chennai,2025-07-11,2,UPI,0.05,SamanthaHull@outlook.com,7415321565,successful
4,794,Gerald Cooper,106,80000,sofa,FURN,offline,Chennai,2025-10-06,2,UPI,0.05,GeraldCooper@gmail.com,9739354201,successful
5,864,Cameron Black,108,15000,Mixer grinder,home appliances,offline,Chennai,2025-12-12,1,Cash,0.05,CameronBlack@yahoo.com,9938169477,successful


In [0]:
print("RETAIL DATA2\n")

print("Rows loaded: ", retail_data2.count())
retail_data2.limit(5).display()

RETAIL DATA2

Rows loaded:  4251


transaction_id,customer_id,customer_name,product_id,price,product_name,category,purchase_location,city,transaction_date,quantity,payment_method,discount,email,phone,payment_status
4001,1112,Jasmine Carlson,106,80000,sofa,Furniture,offline,Delhi,12-18-2025,2,Cash,0.1,JasmineCarlson@gmail.com,8286662057,successful
4002,1125,Jeanne Suarez,105,45000,Tv,Electronics,online,Mumbai,2025-08-07,3,Card,0.1,JeanneSuarez@yahoo.com,6435924212,successful
4003,1379,Anthony Howard,107,50000,Dining table,FURN,offline,Delhi,09-19-2025,2,Cash,0.3,Anthony42@gmail.com,8187928333,successful
4004,1570,John Wilkins,107,50000,DINING TABLE,FURN,offline,Delhi,2025-11-20,2,Cash,0.4,JohnWilkins@outlook.com,6440039629,successful
4005,1433,Joseph Knapp,108,15000,Mixer grinder,home appliances,online,Bangalore,2025-09-29,5,Cash,0.25,JosephKnapp@gmail.com,9284221237,successful


In [0]:
print("PRODUCT DETAILS")

print("Rows loaded: ", product_details.count())
product_details.limit(5).display()

PRODUCT DETAILS
Rows loaded:  10


product_id,product_name,category,price
101,Laptop,Electronics,250000
102,Phone,Electronics,70000
103,Shirt,Clothing,1300
104,Shoes,Clothing,8000
105,TV,Electronics,45000



### Transforming the Product Details table

In [0]:
from pyspark.sql.functions import (
    col, 
    isnan,
    split, 
    when, 
    to_date, 
    sha2, 
    length, 
    trim,
    regexp_replace,
    initcap
)

In [0]:
products = (
    product_details
    .filter(
        col("product_id").isNotNull() &
        col("product_name").isNotNull() &
        col("category").isNotNull() &
        col("price").isNotNull() &
        ~isnan(col("price"))
    )
    .select(
        "product_id",
        "product_name",
        "category",
        "price"
    )
    .distinct()
)

if (
    products.count() == 0
    or products.count() != product_details.count()
    or len(products.columns) != EXPECTED_PRODUCT_DETAILS_COLS
):
    raise Exception("Failed to extract the Product details")

print("Products table is created successfully")

Products table is created successfully


## Creating customers table:

- Extract the data from both the tables, Retail data1 and data2
- Then exclude all the duplicates
- Then perform Hashing on the Password and Phone Number
- Then store the final cleaned customer data in Customers.csv

In [0]:
# 1. Extracting the Raw data from the Retail data1 and data2

# REGEX for Email validation
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"


customers_data1 = (
    retail_data1
    .select(
        "customer_id",
        "customer_name",
        "email",
        "phone"
    )
    .filter(
        col("customer_id").isNotNull() &
        col("customer_name").isNotNull() &
        col("email").isNotNull() &
        col("phone").isNotNull() &

        (trim(col("customer_id")) != "") &
        (trim(col("customer_name")) != "") &
        (trim(col("email")) != "") &

        col("email").rlike(email_regex) &
        (length(regexp_replace(col("phone").cast("string"), r"\D", "")) == 10)
    )
    .distinct()
)

customers_data2 = (
    retail_data2
    .select(
        "customer_id",
        "customer_name",
        "email",
        "phone"
    )
    .filter(
        col("customer_id").isNotNull() &
        col("customer_name").isNotNull() &
        col("email").isNotNull() &
        col("phone").isNotNull() &

        (trim(col("customer_id")) != "") &
        (trim(col("customer_name")) != "") &
        (trim(col("email")) != "") &

        col("email").rlike(email_regex) &
        (length(regexp_replace(col("phone").cast("string"), r"\D", "")) == 10)
    )
    .distinct()
)

In [0]:
if customers_data1.count() == 0 or customers_data2.count() == 0:
    raise Exception("Failed to extract the Customer data")

# 2. Making a customers_raw table to store all the raw data
customers_raw = customers_data1.unionByName(customers_data2)

# ERROR: If the Master table failed to union both the customers table
if customers_raw.count() == 0:
    raise Exception("Failed to union the Customer data into master table")

# ERROR: When the row count doesn't match, Means there's data loss
if customers_raw.count() != (customers_data1.count() + customers_data2.count()):
    raise Exception("Customers data not loaded successfully!")

print("Successfully extracted and loaded data into Master Table")

Successfully extracted and loaded data into Master Table


In [0]:
# 3. Hashing the password and Email
customers_clean = (
    customers_raw
    .withColumn("email_hash", sha2(col("email"), 256))
    .withColumn("phone_hash", sha2(col("phone").cast("string"), 256))
)

if customers_clean.count() == 0 or customers_clean.count() != customers_raw.count():
    raise Exception("Data not cleaned properly")

print("Hashing Completed!")

Hashing Completed!


In [0]:
customers = customers_clean.select(["customer_id","customer_name", "email_hash", "phone_hash"])

if customers.count() == 0 or customers_clean.count() != customers.count():
    raise Exception("Data not cleaned properly")

print("Customers Table is formed successfully")

customers.limit(10).display()

Customers Table is formed successfully


customer_id,customer_name,email_hash,phone_hash
666,Emily Griffith,3cf85c8cca0dce338eb6676a2947ee64cf309b327717a3f470ad6262fcd6547a,d1147e14a158bbb7a5b5da46337744f75fa535bcf3dad0859d4c5086008992ac
725,Jody Martinez,fd863b4f53124ac4aec3a4c710ec37c0b77a7871fb5601bf03bc78e4790dd936,9554f639602ed1c35ff8ce7113ce2485597d2069b729afaa2b0bbdfddafbed76
312,James Morales,7697edf68ba785afc16cc453a09891caef14153124fc1ee23b2c155ae383a092,b34e50ad1aa1b080c21f1731d0013fc83a5983d8d6c191de658861ab3340b1ee
160,Eric Richardson,2b053d537a07f405851d86b836ff78105ab7e9b77d725c7ef4cb43bd97251e33,400b8275a14f086cc0b8c2007ba03e0fc3101055d21f13bf12c068ec6484346f
25,Kenneth Johnson,cf92fc6326e3d28e66b076fcbb5e50750d0adac1f36040f64407d7a795d686da,6434b4d66a6c37fe00ace9435dd871c32aac047e656e3e84d6115cac59e64f00
22,James Walters,ec2d6f294e97b0d6857fb8bbfc52cef910731d7835d27fa3c1b6b9ec432764c2,1b3dbe32c59ae3375826fad2383530e74974517bf1d98aa9144840706a38b032
216,Patricia Myers,df13b2ed4d71af2b15dcec3e7b165d66e4e82acbeb3f02b21338cac7c1861989,036e3652b99457ccad4c62f46bc098c17c0487c7bb213da9a1ffa7940f54cb11
798,Amber Murphy,05a5b0575faa38d0083c52d65db172c373dcbeb1905f9972fc2165cadc42529b,5b371aa060d7e7ce86dfa957a31286cf86e1213ed6815db63cd29e0d5967eead
893,Taylor Walton,62b55ca9bc80c3d8e2b0bf46d326e39f1205292a4ecb43b50d5f1dc64e72ae29,ad1eb1b727a1ffa20e121a38be89b2a9e85bfe644937069065ee3b8e25177628
267,Scott Price,5279839c2ca895760f8cd1fb9474739c48c9d59b9da2013bd8113b53d94eaa8f,4f45f06f2a0bda7f0850b33afb55e5b0bc056daa55cb63f97f557a09cf6cfa81


**Customers table is fomration is completed**

### Now making a Master retail table named as sales

In [0]:
# Loading retail data1
sales_data1 = (
    retail_data1
    .filter(
        col("transaction_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &

        col("quantity").isNotNull() &
        col("discount").isNotNull() &
        col("transaction_date").isNotNull() &

        col("city").isNotNull() &
        col("purchase_location").isNotNull() &
        col("payment_method").isNotNull() &
        col("payment_status").isNotNull() &

        (trim(col("city")) != "") &
        (trim(col("purchase_location")) != "") &
        (trim(col("payment_method")) != "") &
        (trim(col("payment_status")) != "") &
        (trim(col("transaction_date")) != "")
    )
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "quantity",
        "city",
        "transaction_date",
        col("purchase_location").alias("purchase_mode"),
        "payment_method",
        "discount",
        "payment_status"
    )
    .distinct()
)

sales_data2 = (
    retail_data2
    .filter(
        col("transaction_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &

        col("quantity").isNotNull() &
        col("discount").isNotNull() &
        col("transaction_date").isNotNull() &

        col("city").isNotNull() &
        col("purchase_location").isNotNull() &
        col("payment_method").isNotNull() &
        col("payment_status").isNotNull() &

        (trim(col("city")) != "") &
        (trim(col("purchase_location")) != "") &
        (trim(col("payment_method")) != "") &
        (trim(col("payment_status")) != "") &
        (trim(col("transaction_date")) != "")
    )
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "quantity",
        "city",
        "transaction_date",
        col("purchase_location").alias("purchase_mode"),
        "payment_method",
        "discount",
        "payment_status"
    )
    .distinct()
)



# Checking if the data is loaded properly
# ERROR: sales_data1 or sales_data2 is empty
if sales_data1.count() == 0 or sales_data2.count() == 0:
    raise Exception("Failed to extract the Sales data")

sales_raw_data = sales_data1.unionByName(sales_data2)


# Checking for Data load
# ERROR: sales_raw_data is empty
if sales_raw_data.count() != sales_data1.count() + sales_data2.count():
    raise Exception("Data not loaded properly")
print("Sales Data is formed successfully")

Sales Data is formed successfully


In [0]:
sales_raw_data = sales_raw_data.withColumn(
    "valid_quantity",
    when(col("quantity") >= 0, "Valid").otherwise("Invalid")
)
sales_raw_data = sales_raw_data.withColumn(
    "valid_discount",
    when(col("discount") >= 0, "Valid").otherwise("Invalid")
)

In [0]:
sales_raw_data.groupBy("valid_quantity").count().display()
sales_raw_data.groupBy("valid_discount").count().display()

valid_quantity,count
Valid,8429
Invalid,65


valid_discount,count
Valid,8494


In [0]:
sales_raw_data = sales_raw_data.withColumn(
    "formatted_date",
    when(
        split(col("transaction_date"), "-").getItem(0).cast("int") > 31,
        to_date(col("transaction_date"), "yyyy-MM-dd")
    ).when(
        split(col("transaction_date"), "-").getItem(0).cast("int") > 12,
        to_date(col("transaction_date"), "dd-MM-yyyy")
    ).otherwise(
        to_date(col("transaction_date"), "MM-dd-yyyy")
    )
)

if sales_raw_data.count() != sales_data1.count() + sales_data2.count():
    raise Exception("Data not loaded properly")
print("Date is Formatted Successfully successfully")

Date is Formatted Successfully successfully


In [0]:
EXPECTED_COLS = 13

if len(sales_raw_data.columns) != EXPECTED_COLS:
    raise Exception("Failed to create Master sales raw data table!")

print("Sales Raw Data is formed successfully!!")

Sales Raw Data is formed successfully!!


In [0]:
sales_raw_data.limit(10).display()

transaction_id,customer_id,product_id,quantity,city,transaction_date,purchase_mode,payment_method,discount,payment_status,valid_quantity,valid_discount,formatted_date
415,166,104,5,Bangalore,03-18-2026,online,Cash,0.05,successful,Valid,Valid,2026-03-18
463,121,105,2,Delhi,2025-10-25,offline,Cash,0.35,successful,Valid,Valid,2025-10-25
760,680,104,1,Delhi,05-25-2025,offline,Cash,0.4,successful,Valid,Valid,2025-05-25
880,483,109,2,Bangalore,2025-05-18,offline,Cash,0.35,successful,Valid,Valid,2025-05-18
882,97,106,3,Bangalore,2025-01-07,offline,Cash,0.05,successful,Valid,Valid,2025-01-07
902,109,107,3,Mumbai,2025-12-19,offline,Cash,0.35,successful,Valid,Valid,2025-12-19
1189,489,106,5,Hyderabad,2025-05-22,online,Cash,0.1,successful,Valid,Valid,2025-05-22
1276,816,107,2,Delhi,08-21-2025,offline,UPI,0.2,successful,Valid,Valid,2025-08-21
1503,139,109,3,Mumbai,2025-04-23,offline,NetBanking,0.25,successful,Valid,Valid,2025-04-23
2124,183,107,5,Hyderabad,06-13-2025,offline,NetBanking,0.15,successful,Valid,Valid,2025-06-13


In [0]:
sales =  sales_raw_data.select(
    "transaction_id",
    "customer_id",
    "product_id",
    "quantity",
    "valid_quantity",
    "city",
    col("formatted_date").alias("transaction_date"),
    "purchase_mode",
    "payment_method",
    "discount",
    "valid_discount",
    "payment_status"
)

sales = sales.withColumn(
    "payment_status",
    initcap(col("payment_status"))
)

if sales.count() == 0 or sales.count() != sales_raw_data.count() or (len(sales.columns)) != 12:
    raise Exception("Failed to create Sales table!")
print("Successfully created Sales Table!")

Successfully created Sales Table!


In [0]:
sales.display()

In [0]:
# Save Customer data
customers.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{TRANSFORMED_DATA_PATH}/dim_customer")

print("Customers data has been saved")

# Save Product data
products.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{TRANSFORMED_DATA_PATH}/dim_product")

print("Products data has been saved")

# Save Sales data
sales.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{TRANSFORMED_DATA_PATH}/fact_sales")

print("Sales data has been saved")

In [0]:
dbutils.notebook.exit(
    "SUCCESS: Customer Dimension created successfully"
)